# 03 — Biometric GAMs (Fig 3, S6–S13)

Population-level GAM fits of the five core biometrics (RHR, HRV, RR, skin
temp, blood O₂) on cycle day × cycle length × age, with cardiovascular,
sleep, season, and BMI covariates. Reproduces Fig 3 plus seven supplementary
figures.

Mirrors the source notebook (`figure_generator_revision.ipynb`) cells 1, 2,
4, 8, 9, 68, 69, 70 in this F-1a stage. F-1b–F-6 add the plotting and
contrast-printing pieces.

Requires R 4.x with `mgcv` installed and `rpy2` on `PYTHONPATH`. Fitted
models are cached to `models/pct_*.rds`. To refit from scratch, delete the
files in `models/` first.

In [ ]:
try:
    import rpy2  # noqa: F401
    from rpy2.robjects.packages import importr
    importr('mgcv')
except Exception as e:
    raise RuntimeError(
        "Notebook 03 requires rpy2 and R with the `mgcv` package. "
        "Install via `pip install -e .[r]` and ensure R 4.x is on PATH "
        "with mgcv installed (`install.packages('mgcv')`)."
    ) from e

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from menstrual_cycle_analysis import (
    CycleLengthAnalyses,
    PhysioMethods,
    config,
    load_paper_data,
)
from menstrual_cycle_analysis.r_utils import load_gam_models, save_gam_models

np.random.seed(0)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Setup (mirrors source cells 1, 2, 4, 8, 9, 68)

Cell 68 in the source builds a `PhysioBehaviorAnalyses` instance whose
constructor calls a chain of `pm.add_*` helpers as a side effect on `PM`.
We invoke those helpers directly here — same end state, no detour through
the PBA constructor.

In [ ]:
day_df, CBM = load_paper_data()

CBM.add_sleep_behaviors('user')
CBM.add_workout_behaviors('user')
CBM.add_sleep_behaviors('cycle')
CBM.add_workout_behaviors('cycle')

cla = CycleLengthAnalyses(CBM=CBM)

PM = PhysioMethods(cbm=CBM)
PM.get_reference_table()
PM.process_physio_data(overwrite=True)

PM.add_wo_data()
PM.add_sleep_2_ref()
PM.add_wo_2_ref()
PM.add_acute_chronic_behaviors_data()
PM.add_acute_chronic_behaviors_ref()
PM.add_seasonalilty_2_ref()

## Fit GAM models (mirrors source cell 69)

Five `mgcv::bam` models, one per core biometric, with AR(1) within
subject-cycle. Each fit takes ~5–30 minutes; the loop below caches fitted
models and the metrics table to `models/` so subsequent runs are fast.

In [ ]:
PM.gam_additional_covariates = config.GAM_ADDITIONAL_COVARIATES

config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = config.MODELS_DIR / 'gam_models_metrics.parquet'
biometrics = [f'pct_{vv}' for vv in PM.CORE_BIOMETRICS]
cache_complete = (
    all((config.MODELS_DIR / f'{b}.rds').exists() for b in biometrics)
    and metrics_path.exists()
)

if cache_complete:
    print('Cache hit: loading fitted GAMs from models/')
    PM.gam_models = {}
    load_gam_models(PM, config.MODELS_DIR, keys=biometrics)
    PM.gam_models_metrics = pd.read_parquet(metrics_path)
else:
    PM.fit_gam_models(
        additional_covariates=PM.gam_additional_covariates,
        overwrite=False,
    )
    save_gam_models(PM, config.MODELS_DIR, overwrite=False)
    PM.gam_models_metrics.to_parquet(metrics_path)

In [ ]:
print(PM.gam_models_metrics)